# Ridge & Lasso Regression — Diabetes Dataset

**Objective:** Compare Standard Linear Regression, Ridge (L2), and Lasso (L1) regularization on the Diabetes dataset. Tune hyperparameters using cross-validation.

**Methods:**
1. Standard Linear Regression (no regularization — baseline)
2. Ridge Regression (L2 penalty — shrinks coefficients)
3. Lasso Regression (L1 penalty — shrinks + eliminates coefficients)

**Evaluation:** MSE, RMSE, R² Score, Cross-Validation

---
## Cell 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")

---
## Cell 2 — Load the Diabetes Dataset

In [ ]:
# Load Diabetes dataset (built into sklearn — no download needed)
diabetes = load_diabetes()

# Convert to DataFrame
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target  # Disease progression after 1 year

print("=" * 60)
print("DIABETES DATASET")
print("=" * 60)
print(f"\nShape: {df.shape}")
print(f"Samples: {df.shape[0]}")
print(f"Features: {df.shape[1] - 1}")
print(f"\nFeature names: {list(diabetes.feature_names)}")
print(f"\nTarget: Quantitative measure of disease progression one year after baseline")
print(f"\nDataset description:")
print("  - age, sex, bmi, bp: basic patient info")
print("  - s1–s6: six blood serum measurements")
print("  - All features are already mean-centered and scaled by std")
df.head()

---
## Cell 3 — Explore the Dataset

In [ ]:
print("=" * 60)
print("DATASET EXPLORATION")
print("=" * 60)

print(f"\nMissing values: {df.isnull().sum().sum()}")

print("\n--- Feature Statistics ---")
print(df.describe().round(4).to_string())

print("\n--- Target Statistics ---")
print(f"  Mean:   {df['target'].mean():.2f}")
print(f"  Median: {df['target'].median():.2f}")
print(f"  Std:    {df['target'].std():.2f}")
print(f"  Range:  {df['target'].min():.0f} — {df['target'].max():.0f}")

# Correlation with target
print("\n--- Feature Correlation with Target ---")
correlations = df.corr()['target'].drop('target').sort_values(ascending=False)
for feat, corr in correlations.items():
    bar = '█' * int(abs(corr) * 30)
    sign = '+' if corr > 0 else '-'
    print(f"  {feat:>5}: {sign}{abs(corr):.3f}  {bar}")

---
## Cell 4 — Visualize the Data

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

features = diabetes.feature_names
for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df['target'], alpha=0.3, s=10, color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Target' if i % 5 == 0 else '')
    axes[i].set_title(f'{feat} vs Target', fontsize=10)

plt.suptitle('All Features vs Disease Progression Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  - bmi and s5 show the strongest positive linear relationship with target")
print("  - Multiple features are correlated with each other (multicollinearity)")
print("  - This is exactly where Ridge & Lasso shine — they handle multicollinearity!")

---
## Cell 5 — Data Preprocessing

In [ ]:
print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

# ---- Step 1: Separate features and target ----
X = diabetes.data   # Already preprocessed by sklearn (centered & scaled)
y = diabetes.target
print(f"\nStep 1: Features shape = {X.shape}, Target shape = {y.shape}")

# ---- Step 2: Train-Test Split (80/20) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nStep 2: Train-Test Split")
print(f"  Training: {X_train.shape[0]} samples")
print(f"  Testing:  {X_test.shape[0]} samples")

# ---- Step 3: Additional scaling (good practice for regularization) ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"\nStep 3: Feature scaling applied")
print(f"  Note: Although sklearn's diabetes data is already scaled,")
print(f"  we re-scale to ensure zero mean and unit variance on the train split.")
print(f"  This is critical for Ridge & Lasso — they penalize coefficient magnitude,")
print(f"  so features must be on the same scale for fair penalization.")

print("\nPreprocessing complete!")

---
## Cell 6 — Standard Linear Regression (Baseline)

In [ ]:
# ---- Standard Linear Regression (no regularization) ----
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

# Cross-validation
cv_scores_lr = cross_val_score(lr_model, X_train_scaled, y_train,
                                cv=5, scoring='r2')

print("=" * 55)
print("STANDARD LINEAR REGRESSION (Baseline — No Regularization)")
print("=" * 55)
print(f"\nCoefficients: {lr_model.coef_.round(2)}")
print(f"Intercept:    {lr_model.intercept_:.2f}")
print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'MSE':<20} {mean_squared_error(y_train, y_train_pred_lr):>10.2f} {mean_squared_error(y_test, y_test_pred_lr):>10.2f}")
print(f"{'RMSE':<20} {np.sqrt(mean_squared_error(y_train, y_train_pred_lr)):>10.2f} {np.sqrt(mean_squared_error(y_test, y_test_pred_lr)):>10.2f}")
print(f"{'R²':<20} {r2_score(y_train, y_train_pred_lr):>10.4f} {r2_score(y_test, y_test_pred_lr):>10.4f}")
print(f"\n5-Fold CV R²: {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}")

print("\nHuman Analogy: This is like studying everything with equal effort —")
print("no prioritization. Some features may dominate or cause instability.")

---
## Cell 7 — Ridge Regression with Hyperparameter Tuning (Cross-Validation)

In [ ]:
# ---- Ridge Regression: Hyperparameter Tuning using RidgeCV ----

# Define a range of alpha values to search
alphas_ridge = np.logspace(-4, 4, 100)  # 0.0001 to 10000

# RidgeCV does built-in cross-validation to find the best alpha
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)

best_alpha_ridge = ridge_cv.alpha_
print("=" * 55)
print("RIDGE REGRESSION (L2 Regularization)")
print("=" * 55)
print(f"\nBest alpha (λ) found by 5-Fold CV: {best_alpha_ridge:.4f}")

# Train Ridge with best alpha
ridge_model = Ridge(alpha=best_alpha_ridge)
ridge_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_ridge = ridge_model.predict(X_train_scaled)
y_test_pred_ridge = ridge_model.predict(X_test_scaled)

# Cross-validation
cv_scores_ridge = cross_val_score(ridge_model, X_train_scaled, y_train,
                                   cv=5, scoring='r2')

print(f"Coefficients: {ridge_model.coef_.round(2)}")
print(f"Intercept:    {ridge_model.intercept_:.2f}")
print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'MSE':<20} {mean_squared_error(y_train, y_train_pred_ridge):>10.2f} {mean_squared_error(y_test, y_test_pred_ridge):>10.2f}")
print(f"{'RMSE':<20} {np.sqrt(mean_squared_error(y_train, y_train_pred_ridge)):>10.2f} {np.sqrt(mean_squared_error(y_test, y_test_pred_ridge)):>10.2f}")
print(f"{'R²':<20} {r2_score(y_train, y_train_pred_ridge):>10.4f} {r2_score(y_test, y_test_pred_ridge):>10.4f}")
print(f"\n5-Fold CV R²: {cv_scores_ridge.mean():.4f} ± {cv_scores_ridge.std():.4f}")

print(f"\nWhat Ridge does:")
print(f"  Cost = MSE + α × Σ(θᵢ²)")
print(f"  It SHRINKS all coefficients toward zero but never eliminates them.")
print(f"\nHuman Analogy: Like a teacher saying 'don't rely too much on any single topic' —")
print(f"  it keeps all subjects but prevents over-dependence on any one.")

---
## Cell 8 — Lasso Regression with Hyperparameter Tuning (Cross-Validation)

In [ ]:
# ---- Lasso Regression: Hyperparameter Tuning using LassoCV ----

alphas_lasso = np.logspace(-4, 2, 100)  # 0.0001 to 100

# LassoCV does built-in cross-validation
lasso_cv = LassoCV(alphas=alphas_lasso, cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

best_alpha_lasso = lasso_cv.alpha_
print("=" * 55)
print("LASSO REGRESSION (L1 Regularization)")
print("=" * 55)
print(f"\nBest alpha (λ) found by 5-Fold CV: {best_alpha_lasso:.4f}")

# Train Lasso with best alpha
lasso_model = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_lasso = lasso_model.predict(X_train_scaled)
y_test_pred_lasso = lasso_model.predict(X_test_scaled)

# Cross-validation
cv_scores_lasso = cross_val_score(lasso_model, X_train_scaled, y_train,
                                   cv=5, scoring='r2')

print(f"Coefficients: {lasso_model.coef_.round(2)}")
print(f"Intercept:    {lasso_model.intercept_:.2f}")

# Count zero coefficients (Lasso's feature selection)
n_zero = np.sum(lasso_model.coef_ == 0)
n_nonzero = np.sum(lasso_model.coef_ != 0)
print(f"\nFeature Selection:")
print(f"  Non-zero coefficients: {n_nonzero} / {len(lasso_model.coef_)}")
print(f"  Eliminated features:   {n_zero} / {len(lasso_model.coef_)}")

# Show which features were kept/eliminated
print(f"\n  {'Feature':<8} {'Coefficient':>12} {'Status':>10}")
print(f"  {'-'*32}")
for feat, coef in zip(diabetes.feature_names, lasso_model.coef_):
    status = '✓ KEPT' if coef != 0 else '✗ DROPPED'
    print(f"  {feat:<8} {coef:>12.4f} {status:>10}")

print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'MSE':<20} {mean_squared_error(y_train, y_train_pred_lasso):>10.2f} {mean_squared_error(y_test, y_test_pred_lasso):>10.2f}")
print(f"{'RMSE':<20} {np.sqrt(mean_squared_error(y_train, y_train_pred_lasso)):>10.2f} {np.sqrt(mean_squared_error(y_test, y_test_pred_lasso)):>10.2f}")
print(f"{'R²':<20} {r2_score(y_train, y_train_pred_lasso):>10.4f} {r2_score(y_test, y_test_pred_lasso):>10.4f}")
print(f"\n5-Fold CV R²: {cv_scores_lasso.mean():.4f} ± {cv_scores_lasso.std():.4f}")

print(f"\nWhat Lasso does:")
print(f"  Cost = MSE + α × Σ|θᵢ|")
print(f"  It SHRINKS coefficients AND can set some exactly to ZERO (feature selection).")
print(f"\nHuman Analogy: Like a student deciding 'I'll skip irrelevant chapters entirely'")
print(f"  and focus only on the most important topics for the exam.")

---
## Cell 9 — Full Comparison Table

In [ ]:
# Build comparison table
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
    'Alpha (λ)': ['N/A', f'{best_alpha_ridge:.4f}', f'{best_alpha_lasso:.4f}'],
    'Train MSE': [
        mean_squared_error(y_train, y_train_pred_lr),
        mean_squared_error(y_train, y_train_pred_ridge),
        mean_squared_error(y_train, y_train_pred_lasso)
    ],
    'Test MSE': [
        mean_squared_error(y_test, y_test_pred_lr),
        mean_squared_error(y_test, y_test_pred_ridge),
        mean_squared_error(y_test, y_test_pred_lasso)
    ],
    'Train R²': [
        r2_score(y_train, y_train_pred_lr),
        r2_score(y_train, y_train_pred_ridge),
        r2_score(y_train, y_train_pred_lasso)
    ],
    'Test R²': [
        r2_score(y_test, y_test_pred_lr),
        r2_score(y_test, y_test_pred_ridge),
        r2_score(y_test, y_test_pred_lasso)
    ],
    'CV R² (mean)': [
        cv_scores_lr.mean(),
        cv_scores_ridge.mean(),
        cv_scores_lasso.mean()
    ],
    'Non-zero Coefs': [
        np.sum(lr_model.coef_ != 0),
        np.sum(ridge_model.coef_ != 0),
        np.sum(lasso_model.coef_ != 0)
    ]
})

print("=" * 90)
print("FULL COMPARISON: Linear vs Ridge vs Lasso")
print("=" * 90)
print()
print(comparison.to_string(index=False, float_format='%.4f'))

# Find best model
best_idx = comparison['Test R²'].astype(float).idxmax()
print(f"\n>>> Best model by Test R²: {comparison.loc[best_idx, 'Model']} <<<")

---
## Cell 10 — Visualize Coefficient Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(diabetes.feature_names))
width = 0.25

bars1 = ax.bar(x - width, lr_model.coef_, width, label='Linear', color='#3498DB', alpha=0.8)
bars2 = ax.bar(x, ridge_model.coef_, width, label=f'Ridge (α={best_alpha_ridge:.2f})', color='#2ECC71', alpha=0.8)
bars3 = ax.bar(x + width, lasso_model.coef_, width, label=f'Lasso (α={best_alpha_lasso:.2f})', color='#E74C3C', alpha=0.8)

ax.set_xlabel('Features', fontsize=13)
ax.set_ylabel('Coefficient Value', fontsize=13)
ax.set_title('Coefficient Comparison: Linear vs Ridge vs Lasso', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(diabetes.feature_names, fontsize=11)
ax.legend(fontsize=11)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='-')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Linear: Large positive & negative coefficients (potentially unstable)")
print("  - Ridge:  Coefficients shrunk toward zero but ALL are non-zero")
print("  - Lasso:  Some coefficients are EXACTLY zero (automatic feature selection!)")

---
## Cell 11 — Ridge: Effect of Alpha on Coefficients (Regularization Path)

In [ ]:
alphas = np.logspace(-3, 5, 200)

ridge_coefs = []
for a in alphas:
    ridge_temp = Ridge(alpha=a)
    ridge_temp.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge_temp.coef_)

ridge_coefs = np.array(ridge_coefs)

plt.figure(figsize=(12, 6))
for i in range(ridge_coefs.shape[1]):
    plt.plot(alphas, ridge_coefs[:, i], linewidth=1.5, label=diabetes.feature_names[i])

plt.xscale('log')
plt.axvline(x=best_alpha_ridge, color='black', linestyle='--', linewidth=1, label=f'Best α = {best_alpha_ridge:.2f}')
plt.xlabel('Alpha (λ) — Regularization Strength', fontsize=13)
plt.ylabel('Coefficient Value', fontsize=13)
plt.title('Ridge Regularization Path — Coefficients vs Alpha', fontsize=14)
plt.legend(fontsize=9, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("As alpha increases (left → right):")
print("  All coefficients SHRINK toward zero but NEVER become exactly zero.")
print("\nHuman Analogy: Like turning up a 'caution dial' —")
print("  higher alpha = more cautious = simpler model = less risk of overfitting.")

---
## Cell 12 — Lasso: Effect of Alpha on Coefficients (Regularization Path)

In [ ]:
alphas_l = np.logspace(-3, 2, 200)

lasso_coefs = []
for a in alphas_l:
    lasso_temp = Lasso(alpha=a, max_iter=10000)
    lasso_temp.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso_temp.coef_)

lasso_coefs = np.array(lasso_coefs)

plt.figure(figsize=(12, 6))
for i in range(lasso_coefs.shape[1]):
    plt.plot(alphas_l, lasso_coefs[:, i], linewidth=1.5, label=diabetes.feature_names[i])

plt.xscale('log')
plt.axvline(x=best_alpha_lasso, color='black', linestyle='--', linewidth=1, label=f'Best α = {best_alpha_lasso:.2f}')
plt.xlabel('Alpha (λ) — Regularization Strength', fontsize=13)
plt.ylabel('Coefficient Value', fontsize=13)
plt.title('Lasso Regularization Path — Coefficients vs Alpha', fontsize=14)
plt.legend(fontsize=9, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("As alpha increases (left → right):")
print("  Coefficients shrink AND some become EXACTLY ZERO (eliminated!).")
print("  This is Lasso's built-in feature selection.")
print("\nHuman Analogy: Like dropping subjects one by one as exams approach —")
print("  you focus only on what matters most.")

---
## Cell 13 — Cross-Validation Score vs Alpha

In [ ]:
# Compute CV scores for a range of alphas
alphas_cv = np.logspace(-3, 3, 50)

ridge_cv_scores = []
lasso_cv_scores = []

for a in alphas_cv:
    # Ridge CV
    ridge_scores = cross_val_score(Ridge(alpha=a), X_train_scaled, y_train, cv=5, scoring='r2')
    ridge_cv_scores.append(ridge_scores.mean())
    
    # Lasso CV
    lasso_scores = cross_val_score(Lasso(alpha=a, max_iter=10000), X_train_scaled, y_train, cv=5, scoring='r2')
    lasso_cv_scores.append(lasso_scores.mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge CV
axes[0].plot(alphas_cv, ridge_cv_scores, 'o-', color='#2ECC71', linewidth=2, markersize=4)
axes[0].axvline(x=best_alpha_ridge, color='black', linestyle='--', label=f'Best α = {best_alpha_ridge:.4f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (λ)')
axes[0].set_ylabel('CV R² Score')
axes[0].set_title('Ridge: Cross-Validation R² vs Alpha')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Lasso CV
axes[1].plot(alphas_cv, lasso_cv_scores, 'o-', color='#E74C3C', linewidth=2, markersize=4)
axes[1].axvline(x=best_alpha_lasso, color='black', linestyle='--', label=f'Best α = {best_alpha_lasso:.4f}')
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (λ)')
axes[1].set_ylabel('CV R² Score')
axes[1].set_title('Lasso: Cross-Validation R² vs Alpha')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Hyperparameter Tuning — Finding the Best Alpha', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The peak of each curve = optimal alpha.")
print("Too low α = overfitting (too complex). Too high α = underfitting (too simple).")

---
## Cell 14 — Metrics Bar Chart Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = ['Linear', 'Ridge', 'Lasso']
bar_colors = ['#3498DB', '#2ECC71', '#E74C3C']
x_pos = np.arange(len(models))

train_mse = [mean_squared_error(y_train, p) for p in [y_train_pred_lr, y_train_pred_ridge, y_train_pred_lasso]]
test_mse = [mean_squared_error(y_test, p) for p in [y_test_pred_lr, y_test_pred_ridge, y_test_pred_lasso]]
train_r2 = [r2_score(y_train, p) for p in [y_train_pred_lr, y_train_pred_ridge, y_train_pred_lasso]]
test_r2 = [r2_score(y_test, p) for p in [y_test_pred_lr, y_test_pred_ridge, y_test_pred_lasso]]
cv_r2 = [cv_scores_lr.mean(), cv_scores_ridge.mean(), cv_scores_lasso.mean()]

# MSE
axes[0].bar(x_pos - 0.15, train_mse, 0.3, color=bar_colors, alpha=0.5, label='Train')
axes[0].bar(x_pos + 0.15, test_mse, 0.3, color=bar_colors, alpha=1.0, label='Test')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models)
axes[0].set_ylabel('MSE')
axes[0].set_title('Mean Squared Error (lower is better)')
axes[0].legend()

# R²
axes[1].bar(x_pos - 0.15, train_r2, 0.3, color=bar_colors, alpha=0.5, label='Train')
axes[1].bar(x_pos + 0.15, test_r2, 0.3, color=bar_colors, alpha=1.0, label='Test')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(models)
axes[1].set_ylabel('R² Score')
axes[1].set_title('R² Score (higher is better)')
axes[1].legend()

# CV R²
axes[2].bar(x_pos, cv_r2, 0.5, color=bar_colors, alpha=0.85)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(models)
axes[2].set_ylabel('CV R² Score')
axes[2].set_title('5-Fold Cross-Validation R²')
for i, v in enumerate(cv_r2):
    axes[2].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Performance Comparison: Linear vs Ridge vs Lasso', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Cell 15 — Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

predictions = [y_test_pred_lr, y_test_pred_ridge, y_test_pred_lasso]
titles = ['Linear Regression', 'Ridge Regression', 'Lasso Regression']
colors_res = ['#3498DB', '#2ECC71', '#E74C3C']

for i, (pred, title, color) in enumerate(zip(predictions, titles, colors_res)):
    residuals = y_test - pred
    axes[i].scatter(pred, residuals, alpha=0.5, s=20, color=color, edgecolors='white', linewidth=0.3)
    axes[i].axhline(y=0, color='black', linewidth=1, linestyle='--')
    axes[i].set_xlabel('Predicted Values')
    axes[i].set_ylabel('Residuals')
    axes[i].set_title(f'{title}\nResiduals vs Predicted')

plt.suptitle('Residual Analysis — All Three Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Cell 16 — Actual vs Predicted Scatter Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, (pred, title, color) in enumerate(zip(predictions, titles, colors_res)):
    axes[i].scatter(y_test, pred, alpha=0.5, s=20, color=color, edgecolors='white', linewidth=0.3)
    # Perfect prediction line
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    axes[i].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect Prediction')
    axes[i].set_xlabel('Actual Values')
    axes[i].set_ylabel('Predicted Values')
    axes[i].set_title(f'{title}')
    axes[i].legend(fontsize=9)

plt.suptitle('Actual vs Predicted — Closer to the dashed line = better', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Points on the dashed line = perfect prediction.")
print("Spread around the line = prediction error.")

---
## Cell 17 — Summary & Key Takeaways

In [ ]:
print(f"""
╔═══════════════════════════════════════════════════════════════════════════╗
║                       SUMMARY & KEY TAKEAWAYS                            ║
╠═══════════════════════════════════════════════════════════════════════════╣
║                                                                           ║
║  Dataset: Diabetes (442 samples, 10 features)                             ║
║  Target:  Disease progression measure (quantitative)                      ║
║                                                                           ║
║  Models Compared:                                                         ║
║  ┌──────────────────┬───────────────┬────────────────────────────────┐    ║
║  │ Model            │ Regularization│ Key Property                   │    ║
║  ├──────────────────┼───────────────┼────────────────────────────────┤    ║
║  │ Linear Regression│ None          │ No constraint on coefficients  │    ║
║  │ Ridge (L2)       │ α × Σ(θ²)    │ Shrinks all, drops none        │    ║
║  │ Lasso (L1)       │ α × Σ|θ|     │ Shrinks + drops (selection)    │    ║
║  └──────────────────┴───────────────┴────────────────────────────────┘    ║
║                                                                           ║
║  Hyperparameter Tuning:                                                   ║
║  • RidgeCV and LassoCV used with 5-fold cross-validation                 ║
║  • Best Ridge α = {best_alpha_ridge:<10.4f}                                           ║
║  • Best Lasso α = {best_alpha_lasso:<10.4f}                                           ║
║                                                                           ║
║  Key Findings:                                                            ║
║  1. Ridge & Lasso generalize better (lower gap between train & test)     ║
║  2. Lasso performs automatic feature selection (sets coefs to zero)       ║
║  3. Ridge keeps all features but reduces their impact                    ║
║  4. Cross-validation is essential for choosing the right alpha           ║
║                                                                           ║
║  When to Use What:                                                        ║
║  • Linear: When features are independent, no multicollinearity           ║
║  • Ridge:  When all features are useful but correlated                   ║
║  • Lasso:  When you suspect many features are irrelevant                 ║
║                                                                           ║
║  Human Learning Parallels:                                                ║
║  • Linear  = Studying everything equally (no strategy)                   ║
║  • Ridge   = Studying all topics but with balanced effort                ║
║  • Lasso   = Strategically dropping weak topics, acing the important     ║
║  • Alpha   = How strict your study discipline is                         ║
║                                                                           ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")